# 04 — Test vJoy (volant)

Reprise de la méthode `ctypes / vJoyInterface.dll` qui avait fait bouger ton volant. **Aucun mouvement envoyé par défaut.** Tester uniquement voiture à l’arrêt.

In [39]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaEnv').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvre le depot IA_projet_Ouarti_ds_github dans VS Code : son dossier assetto_corsa_gym doit exister.')
PROJECT=REPO/'Code'
if not PROJECT.is_dir():
    raise FileNotFoundError('Dossier Code absent : extraire Code a la racine du depot.')
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
print('Depot :',REPO)
print('Notebooks :',PROJECT)
print('Python :',sys.version.split()[0],sys.executable)


Depot : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github
Notebooks : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code
Python : 3.9.13 c:\Users\elkat\AppData\Local\Programs\Python\Python39\python.exe


In [40]:
import ctypes,time
from pathlib import Path
DLL_PATH=Path(r'C:\Program Files\vJoy\x64\vJoyInterface.dll')
DEVICE_ID=1
HID_USAGE_X=0x30

def get_vjoy():
    if not DLL_PATH.exists():raise FileNotFoundError(str(DLL_PATH)+' : installer vJoy et configurer device 1')
    dll=ctypes.CDLL(str(DLL_PATH))
    dll.GetVJDStatus.argtypes=[ctypes.c_uint];dll.GetVJDStatus.restype=ctypes.c_int
    dll.AcquireVJD.argtypes=[ctypes.c_uint];dll.AcquireVJD.restype=ctypes.c_bool
    dll.SetAxis.argtypes=[ctypes.c_long,ctypes.c_uint,ctypes.c_uint];dll.SetAxis.restype=ctypes.c_bool
    dll.RelinquishVJD.argtypes=[ctypes.c_uint];dll.RelinquishVJD.restype=None
    return dll


In [41]:
# Verification sans mouvement
if DLL_PATH.exists():
    dll=get_vjoy()
    print('DLL vJoy trouvee. Statut device 1 :',dll.GetVJDStatus(DEVICE_ID))
else:
    print('DLL absente :',DLL_PATH)


DLL vJoy trouvee. Statut device 1 : 1


In [42]:
EXECUTER_MOUVEMENTS=True  # Passer a True uniquement si tu confirmes que le jeu est en securite
VALEURS=[0,16384,32767,16384]
if not EXECUTER_MOUVEMENTS:
    print('Simulation seulement, axe X :',VALEURS)
else:
    dll=get_vjoy()
    status=dll.GetVJDStatus(DEVICE_ID)
    if status!=1:raise RuntimeError('Device 1 deja utilise ou indisponible : code '+str(status))
    if not dll.AcquireVJD(DEVICE_ID):raise RuntimeError('Impossible d acquerir le device 1')
    try:
        for x in VALEURS:
            if not dll.SetAxis(int(x),DEVICE_ID,HID_USAGE_X):
                raise RuntimeError('SetAxis False a '+str(x))
            print('Axe X :',x)
            time.sleep(.7)
    finally:
        dll.SetAxis(16384,DEVICE_ID,HID_USAGE_X)
        dll.RelinquishVJD(DEVICE_ID)
        print('Volant recentre et device libere.')


Axe X : 0


Axe X : 16384
Axe X : 32767
Axe X : 16384
Volant recentre et device libere.


In [43]:

import time

# Configuration vJoy
DEVICE_ID = 1

HID_USAGE_X = 0x30  # Direction
HID_USAGE_Y = 0x31  # Accélérateur
HID_USAGE_Z = 0x32  # Frein

# Valeurs vJoy : 0 à 32767
CENTRE = 16384
PEDALE_RELACHEE = 0

# Séquence de direction
VALEURS = [CENTRE, 0, CENTRE, 32767, CENTRE]

# Sécurité
EXECUTER_MOUVEMENTS = True

dll = get_vjoy()

if EXECUTER_MOUVEMENTS:
    if dll.GetVJDStatus(DEVICE_ID) != 1:
        raise RuntimeError("vJoy Device 1 indisponible")

    if not dll.AcquireVJD(DEVICE_ID):
        raise RuntimeError("Impossible d'acquérir vJoy")

    def envoyer(axe, valeur):
        if not dll.SetAxis(valeur, DEVICE_ID, axe):
            raise RuntimeError(f"Echec axe {hex(axe)}")

    try:
        # Initialisation du volant et des pédales
        envoyer(HID_USAGE_X, CENTRE)
        envoyer(HID_USAGE_Y, PEDALE_RELACHEE)
        envoyer(HID_USAGE_Z, PEDALE_RELACHEE)

        print("Accélérateur : relâché (commande 0)")
        print("Frein : relâché (commande 0)")

        # Test volant uniquement
        for valeur in VALEURS:
            envoyer(HID_USAGE_Y, PEDALE_RELACHEE)
            envoyer(HID_USAGE_Z, PEDALE_RELACHEE)
            envoyer(HID_USAGE_X, valeur)

            print("Direction :", valeur)
            time.sleep(0.7)

    finally:
        # Remise au repos
        envoyer(HID_USAGE_X, CENTRE)
        envoyer(HID_USAGE_Y, PEDALE_RELACHEE)
        envoyer(HID_USAGE_Z, PEDALE_RELACHEE)

        dll.RelinquishVJD(DEVICE_ID)
        print("Test terminé, vJoy libéré.")


Accélérateur : relâché (commande 0)
Frein : relâché (commande 0)
Direction : 16384
Direction : 0
Direction : 16384
Direction : 32767
Direction : 16384
Test terminé, vJoy libéré.


In [44]:
import pandas as pd
from pathlib import Path

# Recherche du fichier de télémétrie
CWD = Path.cwd().resolve()

PROJECT = next(
    (
        p for p in [CWD, *CWD.parents]
        if (p / "reports" / "telemetrie_test.csv").is_file()
    ),
    None
)

if PROJECT is None:
    raise FileNotFoundError("Impossible de retrouver telemetrie_test.csv")

# Chargement du CSV
df = pd.read_csv(PROJECT / "reports" / "telemetrie_test.csv")

# Recherche des variables liées aux pédales
colonnes = [
    c for c in df.columns
    if any(mot in c.lower() for mot in ["gas", "throttle", "brake"])
]

print("Variables trouvées :", colonnes)

if colonnes:
    display(df[colonnes].describe())
else:
    print("Aucune variable de pédale trouvée.")

Variables trouvées : ['brakeStatus']


,brakeStatus
count,277.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


In [45]:

import pandas as pd
from pathlib import Path

# 1. Rechercher automatiquement le fichier de télémétrie
CWD = Path.cwd().resolve()

PROJECT = next(
    (
        p for p in [CWD, *CWD.parents]
        if (p / "reports" / "telemetrie_test.csv").is_file()
    ),
    None
)

if PROJECT is None:
    raise FileNotFoundError("Impossible de trouver telemetrie_test.csv")

# 2. Charger les données
df = pd.read_csv(PROJECT / "reports" / "telemetrie_test.csv")

# 3. Identifier les variables des pédales
def trouver_capteur(noms):
    for nom in noms:
        for colonne in df.columns:
            if colonne.lower() == nom.lower():
                return colonne
    return None

accelerateur = trouver_capteur(
    ["accStatus", "gas", "throttle", "accelerator"]
)

frein = trouver_capteur(
    ["brakeStatus", "brake", "brakePedal"]
)

# 4. Afficher les résultats
print("=== COMMANDES DES PÉDALES ===")

capteurs = {
    "Accélérateur": accelerateur,
    "Frein": frein
}

for nom, colonne in capteurs.items():
    print(f"\n--- {nom} ---")

    if colonne is None:
        print("Capteur introuvable dans le CSV")
        continue

    valeurs = pd.to_numeric(df[colonne], errors="coerce").dropna()

    if valeurs.empty:
        print("Aucune valeur numérique disponible")
        continue

    print("Variable :", colonne)
    print("Première valeur :", valeurs.iloc[0])
    print("Dernière valeur :", valeurs.iloc[-1])
    print("Minimum :", valeurs.min())
    print("Maximum :", valeurs.max())
    print("Moyenne :", round(valeurs.mean(), 4))

# 5. Afficher les mesures des deux pédales
colonnes = [c for c in capteurs.values() if c is not None]

if colonnes:
    print("\n=== DERNIÈRES MESURES ENREGISTRÉES ===")
    display(df[colonnes].tail(10))


=== COMMANDES DES PÉDALES ===

--- Accélérateur ---
Variable : accStatus
Première valeur : 0.4200000166893005
Dernière valeur : 0.4200000166893005
Minimum : 0.4200000166893005
Maximum : 0.4200000166893005
Moyenne : 0.42

--- Frein ---
Variable : brakeStatus
Première valeur : 0.0
Dernière valeur : 0.0
Minimum : 0.0
Maximum : 0.0
Moyenne : 0.0

=== DERNIÈRES MESURES ENREGISTRÉES ===


,accStatus,brakeStatus
267,0.42,0.0
268,0.42,0.0
269,0.42,0.0
270,0.42,0.0
271,0.42,0.0
272,0.42,0.0
273,0.42,0.0
274,0.42,0.0
275,0.42,0.0
276,0.42,0.0
